In [2]:
import pandas as pd

interactions = pd.read_parquet("data\\interactions.parquet")

interactions.head(10)

,user_idx,movie_idx,rating
0,0,1104,5.0
1,0,639,3.0
2,0,853,3.0
3,0,3177,4.0
4,0,2162,5.0
5,0,1107,3.0
6,0,1195,5.0
7,0,2599,5.0
8,0,580,4.0
9,0,858,4.0


In [ ]:
def recommend_popular(
        user_idx: int,
        interactions: pd.DataFrame,
        k: int = 10
):
    popularity = (
        interactions.groupby("movie_idx").rating
        .agg(
            count="count",
            mean="mean"
        )
        .reset_index()
        .sort_values(by=["count", "mean"], ascending=False)
    )

    watched_movies = set(
        interactions.loc[
            interactions.user_idx == user_idx, "movie_idx"
        ]
    )

    

    return (
        popularity[
            ~popularity.movie_idx.isin(watched_movies)
        ]
        .head(k)
    )

print(recommend_popular(user_idx=3, interactions=interactions, k=10))


      movie_idx  count      mean
2651       2651   3428  4.317386
575         575   2649  4.058513
2374       2374   2590  4.315830
1178       1178   2583  3.990321
579         579   2578  4.351823
1449       1449   2538  3.739953
593         593   2513  4.254676
2557       2557   2459  4.406263
106         106   2443  4.234957
2203       2203   2369  4.127480


In [ ]:
def bayesian_popularity(
    user_idx: int,
    interactions: pd.DataFrame,
    k: int = 10,
    m: int = 100
):
    C = interactions.rating.mean()

    popularity = (
        interactions.groupby("movie_idx").rating
        .agg(
            count="count",
            mean="mean"
        )
        .reset_index()
    )

    popularity["score"] = (
        popularity.count 
        / (popularity.count + m) 
        * popularity.mean 
        + m 
        / (popularity.count + m)
        * C
    )

    return popularity.sort_values(by="score", ascending=False)

def recommend_bayesian_popularity(
    user_idx: int,
    popularity: pd.DataFrame,
    interactions: pd.DataFrame,
    k: int = 10
):
    
    watched_movies = set(
        interactions.loc[
            interactions.user_idx == user_idx, "movie_idx"
        ]
    )

    return (
        popularity[
            ~popularity.movie_idx.isin(watched_movies)
        ]
        .head(k)
    )
